# Tanpa gold / weak supervision

Ukuran sukses awal adalah coverage, konflik dan audit precision pada gold independen. Skor melawan pseudo-label sendiri bukan generalisasi.

Kode lengkap dari pipeline dan snippet asli. Ikuti urutan tahap dan ubah konfigurasi awal sesuai data. Alternatif tersedia pada cheatsheet utama.

## Setup & konfigurasi

Satu-satunya sel yang wajib diubah. Jalankan paling awal.

In [ ]:
import os
os.environ.update(HF_HUB_OFFLINE="1", TRANSFORMERS_OFFLINE="1", TOKENIZERS_PARALLELISM="false")
from pathlib import Path
import importlib.metadata as md

def env_report(models_dir=None):
    import torch
    for p in ("torch", "transformers", "scikit-learn", "pandas", "numpy", "sentence-transformers", "lightgbm"):
        try:
            print(p, md.version(p))
        except md.PackageNotFoundError:
            print(p, "TIDAK ADA")
    gpu = torch.cuda.is_available()
    print("cuda", gpu, torch.cuda.get_device_name(0) if gpu else "")
    if gpu:
        print("VRAM GB", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), "| bf16", torch.cuda.is_bf16_supported())
    if models_dir and Path(models_dir).exists():
        print("model lokal:", sorted(p.name for p in Path(models_dir).iterdir() if p.is_dir()))
    return gpu

import os
import random
import numpy as np
import torch

def seed_everything(seed=42):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)

# ===== jalankan =====
import re, warnings
import numpy as np
import pandas as pd
warnings.filterwarnings("ignore")
MODELS = Path(os.environ.get("COMP_ROOT", r"C:\gemastik")) / "models"
SEED = 42
seed_everything(SEED)
env_report(MODELS)

DATA, TEXT = "train.csv", "text"
GOLD = "gold"                                    # kolom label manual kecil, -1 bila kosong
LEXICON = {"negatif": ["basi", "buruk", "kecewa", "jelek"], "positif": ["enak", "bagus", "puas", "suka"]}
VEC, MODEL = "tfidf_word", "lr"


## Muat data

Baca file, otomatis mencoba encoding, simpan parquet.

In [ ]:
from pathlib import Path
import pandas as pd

def read_any(path, cache=True, **kw):
    """df = read_any('train.csv'). Lain kali otomatis membaca train.parquet."""
    p = Path(path)
    if p.suffix == ".parquet":
        return pd.read_parquet(p, **kw)
    if cache and p.with_suffix(".parquet").exists():
        return pd.read_parquet(p.with_suffix(".parquet"))
    if p.suffix in (".jsonl", ".json"):
        return pd.read_json(p, lines=p.suffix == ".jsonl", **kw)
    if p.suffix in (".xlsx", ".xls"):
        return pd.read_excel(p, **kw)
    sep = kw.pop("sep", "\t" if p.suffix == ".tsv" else ",")
    for enc in ("utf-8", "utf-8-sig", "latin-1"):
        try:
            df = pd.read_csv(p, sep=sep, encoding=enc, on_bad_lines="warn", **kw)
            break
        except UnicodeDecodeError:
            continue
    if cache:
        df.to_parquet(p.with_suffix(".parquet"), index=False)
    return df

# ===== jalankan =====
df = read_any(DATA)
print(df.shape)


## Preprocessing

Ganti preset bila perlu.

In [ ]:
import html
import re
import unicodedata

SLANG = {"yg": "yang", "gk": "tidak", "ga": "tidak", "gak": "tidak", "nggak": "tidak", "tdk": "tidak", "bgt": "banget", "tp": "tapi", "klo": "kalau", "kalo": "kalau",
         "udh": "sudah", "dah": "sudah", "dpt": "dapat", "jg": "juga", "jd": "jadi", "bnyk": "banyak", "karna": "karena", "krn": "karena", "utk": "untuk", "jgn": "jangan",
         "org": "orang", "dr": "dari", "lgi": "lagi", "lg": "lagi", "bnr": "benar", "sdh": "sudah", "dgn": "dengan", "aja": "saja"}
STOPWORDS = set("""ada adalah agar akan aku anda antara apa apakah atau bagi bahwa banyak beberapa begitu bila bisa dalam dan dari dengan di dia dua hal hanya harus ia ini itu
jadi jika juga kami kamu karena ke kepada ketika kita lagi lain lebih maka masih mereka mungkin nya oleh pada para saat saja sama sampai sangat satu saya sebagai sebuah sedang
sehingga sekali selalu semua seperti setelah sudah supaya tapi telah tentang tersebut tetapi untuk yaitu yang""".split())   # sengaja tanpa tidak, bukan, jangan, belum, kurang
NEGATION = "tidak|tak|bukan|jangan|belum|kurang|tanpa"
_MOJIBAKE = re.compile("[\u00c3\u00c2\u00e2\u00f0]")

PRESETS = {
    # keep = biarkan, token = ganti penanda, remove = hapus
    "none":        dict(unicode=False, html=False, url="keep", email="keep", phone="keep", mention="keep", hashtag="keep", emoji="keep", number="keep", lower=False, repeat=False, punct="keep", slang=False, negation=False, stopword=False),
    "tfidf":       dict(unicode=True, html=True, url="token", email="token", phone="token", mention="token", hashtag="keep", emoji="keep", number="keep", lower=True, repeat=True, punct="keep", slang=False, negation=False, stopword=False),
    "social":      dict(unicode=True, html=True, url="token", email="token", phone="token", mention="token", hashtag="word", emoji="text", number="keep", lower=True, repeat=True, punct="keep", slang=True, negation=False, stopword=False),
    "topic":       dict(unicode=True, html=True, url="remove", email="remove", phone="remove", mention="remove", hashtag="word", emoji="remove", number="remove", lower=True, repeat=True, punct="remove", slang=True, negation=False, stopword=True),
    "transformer": dict(unicode=True, html=True, url="keep", email="keep", phone="keep", mention="keep", hashtag="keep", emoji="keep", number="keep", lower=False, repeat=False, punct="keep", slang=False, negation=False, stopword=False),
    "indobertweet": dict(unicode=True, html=True, url="HTTPURL", email="keep", phone="keep", mention="@USER", hashtag="keep", emoji="text", number="keep", lower=True, repeat=False, punct="keep", slang=False, negation=False, stopword=False),
}

def _sub(text, pattern, how, token):
    return text if how == "keep" else re.sub(pattern, " " if how == "remove" else f" {token if how == 'token' else how} ", text)

def preprocess(text, preset="tfidf", slang_dict=SLANG, stopwords=STOPWORDS, **override):
    """preprocess('Halo http://a.id', 'social'). Ubah satu langkah: preprocess(t, 'tfidf', emoji='remove')."""
    c = {**PRESETS[preset], **override}
    t = "" if text is None or text != text else str(text)
    if c["unicode"]:
        if _MOJIBAKE.search(t):                         # teks rusak encoding
            import ftfy
            t = ftfy.fix_text(t)
        t = unicodedata.normalize("NFKC", t).replace("​", "")
    if c["html"]:
        t = re.sub(r"<[^>]+>", " ", html.unescape(t))
    t = _sub(t, r"[\w.+-]+@[\w-]+\.[\w.-]+", c["email"], "EMAIL")          # email sebelum mention
    t = _sub(t, r"https?://\S+|www\.\S+", c["url"], "URL")
    t = _sub(t, r"(?<!\d)(?:\+?62|0)8\d{7,11}(?!\d)", c["phone"], "TELEPON")
    t = _sub(t, r"@\w+", c["mention"], "USER")
    if c["hashtag"] == "word":
        t = re.sub(r"#(\w+)", r" \1 ", t)
    elif c["hashtag"] != "keep":
        t = _sub(t, r"#\w+", c["hashtag"], "HASHTAG")
    if c["emoji"] != "keep":
        import emoji
        if c["emoji"] == "id":                          # emoji jadi kata Indonesia
            from indoNLP.preprocessing import emoji_to_words
            t = emoji_to_words(t, delimiter=(" ", " "))
        else:
            t = emoji.demojize(t, delimiters=(" ", " ")) if c["emoji"] == "text" else emoji.replace_emoji(t, " " if c["emoji"] == "remove" else " EMOJI ")
    t = _sub(t, r"\d+(?:[.,]\d+)*", c["number"], "ANGKA")
    if c["lower"]:
        keep = {m: m for m in ("HTTPURL", "@USER", "URL", "EMAIL", "TELEPON", "USER", "ANGKA", "EMOJI", "HASHTAG") if m in t}
        t = t.lower()
        for m in keep:
            t = re.sub(r"(?<!\w)" + re.escape(m.lower()) + r"(?!\w)", m, t)
    if c["repeat"]:
        t = re.sub(r"(.)\1{2,}", r"\1\1", t)
    if c["punct"] == "remove":
        t = re.sub(r"[^\w\s]", " ", t)
    if c["slang"]:
        t = re.sub(r"\b\w+\b", lambda m: slang_dict.get(m.group(0).lower(), m.group(0)), t)
    if c["negation"]:
        t = re.sub(rf"\b({NEGATION})\s+(\w+)", r"\1_\2", t, flags=re.I)
    if c["stopword"]:
        t = " ".join(w for w in t.split() if w.lower() not in stopwords)
    return re.sub(r"\s+", " ", t).strip()

def preprocess_corpus(texts, preset="tfidf", stem=False, slang_csv=None, **override):
    """Versi cepat: teks identik hanya diproses sekali, stem per kata unik. Kembalikan list."""
    slang = dict(SLANG)
    if slang_csv:
        import pandas as pd
        lex = pd.read_csv(slang_csv)
        slang.update(dict(zip(lex["slang"].astype(str), lex["formal"].astype(str))))
    texts = list(texts)
    cache = {t: preprocess(t, preset, slang_dict=slang, **override) for t in {x if x == x else None for x in texts}}
    out = [cache[t if t == t else None] for t in texts]
    if stem:
        from Sastrawi.Stemmer.StemmerFactory import StemmerFactory
        st = StemmerFactory().create_stemmer()
        m = {w: st.stem(w) for w in {w for d in out for w in d.split()}}
        out = [" ".join(m[w] for w in d.split()) for d in out]
    return out

def text_versions(df, text="text", slang_csv=None):
    """Tiga versi agar efek normalisasi bisa diuji: text_raw, text_clean (makna utuh), text_norm (slang dinormalkan)."""
    d = df.copy()
    d["text_raw"] = d[text]
    d["text_clean"] = preprocess_corpus(d[text], "tfidf")
    d["text_norm"] = preprocess_corpus(d[text], "social", slang_csv=slang_csv)
    return d

# ===== jalankan =====
texts = preprocess_corpus(df[TEXT], "tfidf")


## Aturan menjadi label

Audit tiap aturan.

In [ ]:
import re
import numpy as np
import pandas as pd

def keyword_lf(texts, lexicon):
    """lexicon: {nama: [kata]}. Kembalikan matriks hit (n, n_nama) dengan pencocokan batas kata."""
    names = list(lexicon)
    pats = [re.compile(r"\b(?:" + "|".join(map(re.escape, sorted(lexicon[n], key=len, reverse=True))) + r")\b", re.I) for n in names]
    return pd.DataFrame([[len(p.findall(str(t))) for p in pats] for t in texts], columns=names)

def audit_lf(H, texts, k=30, seed=42):
    """Jumlah hit dan sampel acak untuk dibaca."""
    print(H.gt(0).mean().round(4).to_string())
    return {c: pd.Series(list(texts))[H[c] > 0].sample(min(k, int((H[c] > 0).sum())), random_state=seed).tolist() for c in H}

def vote_lf(H, weights=None, abstain=-1):
    """Voting tertimbang. Baris tanpa hit menjadi abstain. weights dari presisi aturan pada gold."""
    A = H.to_numpy(float) * (np.asarray(weights) if weights is not None else 1.0)
    return np.where(A.sum(1) > 0, A.argmax(1), abstain)

def lf_accuracy(pred, gold, abstain=-1):
    pred, gold = np.asarray(pred), np.asarray(gold)
    c = pred != abstain
    return dict(coverage=round(c.mean(), 4), precision=round(float((pred[c] == gold[c]).mean()) if c.any() else 0, 4))

# ===== jalankan =====
H = keyword_lf(texts, LEXICON)
samples = audit_lf(H, texts, 30)
weak = vote_lf(H)
print("cakupan", round(float((weak != -1).mean()), 3))


## Latih classifier

Generalisasi di luar kata kunci.

In [ ]:
import os
import re
import numpy as np
import pandas as pd
from scipy import sparse
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.decomposition import TruncatedSVD
from sklearn.feature_extraction.text import CountVectorizer, HashingVectorizer, TfidfVectorizer
from sklearn.pipeline import make_pipeline, make_union
from sklearn.preprocessing import FunctionTransformer, Normalizer

def _tokens(t):
    return re.findall(r"\w+", str(t).lower())

def _meta(texts):
    s = pd.Series(list(texts)).fillna("").astype(str)
    n = s.str.len().clip(lower=1)
    return sparse.csr_matrix(pd.DataFrame({"len": np.log1p(n), "words": np.log1p(s.str.split().str.len()), "upper": s.str.count(r"[A-Z]") / n, "digit": s.str.count(r"\d") / n,
                                           "punct": s.str.count(r"[^\w\s]") / n, "excl": s.str.count("!").clip(upper=5), "url": s.str.count(r"https?://|URL"), "mention": s.str.count(r"@\w+|USER")}).to_numpy("float32"))

class RawText(BaseEstimator, TransformerMixin):
    """Teks mentah untuk model neural."""
    def fit(self, X, y=None): return self
    def transform(self, X): return list(X)

class GensimVec(BaseEstimator, TransformerMixin):
    """kind: w2v, fasttext, doc2vec. idf=True: rata-rata berbobot."""
    def __init__(self, kind="w2v", dim=200, window=5, min_count=2, epochs=5, idf=False, workers=4, seed=42):
        self.kind, self.dim, self.window, self.min_count, self.epochs, self.idf, self.workers, self.seed = kind, dim, window, min_count, epochs, idf, workers, seed
    def fit(self, X, y=None):
        from gensim.models import FastText, Word2Vec
        from gensim.models.doc2vec import Doc2Vec, TaggedDocument
        toks = [_tokens(t) for t in X]
        kw = dict(vector_size=self.dim, min_count=self.min_count, epochs=self.epochs, workers=self.workers, seed=self.seed)
        if self.kind == "doc2vec":
            self.m_ = Doc2Vec([TaggedDocument(t, [i]) for i, t in enumerate(toks)], dm=0, **kw)
        else:
            self.m_ = (FastText if self.kind == "fasttext" else Word2Vec)(toks, window=self.window, **kw).wv
        self.w_ = None
        if self.idf:
            t = TfidfVectorizer(token_pattern=r"\w+").fit(X)
            self.w_ = dict(zip(t.get_feature_names_out(), t.idf_))
        return self
    def transform(self, X):
        out = np.zeros((len(X), self.dim), dtype="float32")
        for i, t in enumerate(X):
            w = _tokens(t)
            if self.kind == "doc2vec":
                out[i] = self.m_.infer_vector(w)
                continue
            w = w if self.kind == "fasttext" else [x for x in w if x in self.m_]
            if w:
                a = np.array([self.w_.get(x, 1.0) for x in w])[:, None] if self.w_ else np.ones((len(w), 1))
                out[i] = (self.m_[w] * a).sum(0) / a.sum()
        return out

class EmbedVec(BaseEstimator, TransformerMixin):
    """Embedding SentenceTransformer dengan cache per teks agar tiap fold tidak encode ulang."""
    _cache = {}
    def __init__(self, model_path=None, prefix="", batch=128):
        self.model_path, self.prefix, self.batch = model_path, prefix, batch
    def fit(self, X, y=None): return self
    def transform(self, X):
        c = EmbedVec._cache.setdefault(str(self.model_path), {})
        miss = [t for t in dict.fromkeys(map(str, X)) if t not in c]
        if miss:
            from sentence_transformers import SentenceTransformer
            m = SentenceTransformer(str(self.model_path), local_files_only=True)
            for t, v in zip(miss, m.encode([self.prefix + t for t in miss], batch_size=self.batch, normalize_embeddings=True, show_progress_bar=False)):
                c[t] = v.astype("float32")
        return np.vstack([c[str(t)] for t in X])

def make_vectorizer(kind="tfidf_union", **kw):
    """Semua hasilnya punya fit_transform dan transform."""
    word = dict(ngram_range=(1, 2), min_df=2, max_df=.95, sublinear_tf=True, dtype="float32", token_pattern=r"(?u)\b\w+\b")
    char = dict(analyzer="char_wb", ngram_range=(2, 5), min_df=3, max_features=500_000, sublinear_tf=True, dtype="float32")
    if kind == "tfidf_word": return TfidfVectorizer(**{**word, **kw})
    if kind == "tfidf_char": return TfidfVectorizer(**{**char, **kw})
    if kind == "tfidf_union": return make_union(TfidfVectorizer(**word), TfidfVectorizer(**char))
    if kind == "count": return CountVectorizer(**{**dict(ngram_range=(1, 1), min_df=2, max_df=.95), **kw})
    if kind == "binary": return CountVectorizer(**{**dict(ngram_range=(1, 2), min_df=2, binary=True), **kw})
    if kind == "hashing": return HashingVectorizer(**{**dict(n_features=2 ** 20, alternate_sign=False, ngram_range=(1, 2), dtype="float32"), **kw})
    if kind == "lsa": return make_pipeline(TfidfVectorizer(**word), TruncatedSVD(kw.pop("k", 256), random_state=42), Normalizer(copy=False))
    if kind == "meta": return FunctionTransformer(_meta, validate=False)
    if kind == "tfidf_meta": return make_union(TfidfVectorizer(**word), FunctionTransformer(_meta, validate=False))
    if kind in ("w2v", "fasttext", "doc2vec"): return GensimVec(kind, **kw)
    if kind == "w2v_idf": return GensimVec("w2v", idf=True, **kw)
    if kind == "embed": return EmbedVec(**kw)
    if kind == "raw": return RawText()
    raise ValueError(f"vectorizer tidak dikenal: {kind}")

import re
from collections import Counter
from functools import partial
import numpy as np
from sklearn.base import BaseEstimator, ClassifierMixin

def _net(kind, vocab, n_out, dim=200, hidden=128, drop=.3):
    import torch
    import torch.nn as nn
    class Net(nn.Module):
        def __init__(self):
            super().__init__()
            self.emb, self.drop = nn.Embedding(vocab, dim, padding_idx=0), nn.Dropout(drop)
            if kind == "cnn":
                self.conv = nn.ModuleList(nn.Conv1d(dim, hidden, k, padding=k // 2) for k in (3, 4, 5))
                self.out = nn.Linear(hidden * 3, n_out)
            else:
                bi = kind in ("bigru", "bilstm", "attn")
                self.rnn = (nn.LSTM if "lstm" in kind else nn.GRU)(dim, hidden, batch_first=True, bidirectional=bi)
                self.att = nn.Linear(hidden * (2 if bi else 1), 1)
                self.out = nn.Linear(hidden * (2 if bi else 1), n_out)
        def forward(self, x):
            e = self.emb(x)
            if kind == "cnn":
                return self.out(self.drop(torch.cat([c(e.transpose(1, 2)).relu().amax(2) for c in self.conv], 1)))
            h, _ = self.rnn(e)
            w = self.att(h).squeeze(-1).masked_fill(x == 0, -1e4).softmax(1) if kind == "attn" else (x != 0).float() / (x != 0).sum(1, keepdim=True).clamp(min=1)
            return self.out(self.drop((h * w.unsqueeze(-1)).sum(1)))
    return Net()

class NeuralText(BaseEstimator, ClassifierMixin):
    """Terima list teks. Pakai bersama make_vectorizer('raw')."""
    def __init__(self, kind="cnn", epochs=3, batch=128, lr=1e-3, max_len=128, min_count=2, device=None, seed=42):
        self.kind, self.epochs, self.batch, self.lr, self.max_len, self.min_count, self.device, self.seed = kind, epochs, batch, lr, max_len, min_count, device, seed
    def _ids(self, T):
        a = np.zeros((len(T), self.max_len), dtype="int64")
        for i, t in enumerate(T):
            w = [self.v_.get(x, 1) for x in re.findall(r"\w+", str(t).lower())[:self.max_len]]
            a[i, :len(w)] = w
        return a
    def fit(self, T, y):
        import torch
        torch.manual_seed(self.seed)
        self.d_ = self.device or ("cuda" if torch.cuda.is_available() else "cpu")
        c = Counter(w for t in T for w in re.findall(r"\w+", str(t).lower()))
        self.v_ = {w: i + 2 for i, (w, k) in enumerate(c.most_common()) if k >= self.min_count}
        self.classes_, yy = np.unique(y, return_inverse=True)
        X, yy = torch.tensor(self._ids(T)), torch.tensor(yy)
        self.n_ = _net(self.kind, len(self.v_) + 2, len(self.classes_)).to(self.d_)
        opt, loss = torch.optim.AdamW(self.n_.parameters(), lr=self.lr), torch.nn.CrossEntropyLoss()
        for _ in range(self.epochs):
            self.n_.train()
            for b in torch.randperm(len(yy)).split(self.batch):
                opt.zero_grad(); loss(self.n_(X[b].to(self.d_)), yy[b].to(self.d_)).backward(); opt.step()
        return self
    def predict_proba(self, T):
        import torch
        self.n_.eval()
        with torch.inference_mode():
            return np.vstack([self.n_(b.to(self.d_)).softmax(1).float().cpu().numpy() for b in torch.tensor(self._ids(T)).split(512)])
    def predict(self, T):
        return self.classes_[self.predict_proba(T).argmax(1)]

def make_model(name="lr", seed=42, n_jobs=8, **kw):
    if name in ("cnn", "gru", "bigru", "lstm", "bilstm", "attn"):
        return NeuralText(name, seed=seed, **kw)
    from sklearn.linear_model import LogisticRegression, Ridge, RidgeClassifier, SGDClassifier
    if name == "lr": return LogisticRegression(**{**dict(C=4.0, max_iter=2000), **kw})
    if name == "sgd": return SGDClassifier(**{**dict(loss="log_loss", alpha=1e-6, random_state=seed), **kw})
    if name == "ridge_clf": return RidgeClassifier(**kw)
    if name in ("svc", "svc_cal"):
        from sklearn.svm import LinearSVC
        m = LinearSVC(**{**dict(C=.5, dual="auto", max_iter=5000), **kw})
        if name == "svc":
            return m
        from sklearn.calibration import CalibratedClassifierCV
        return CalibratedClassifierCV(m, method="sigmoid", cv=3)
    if name in ("cnb", "mnb"):
        from sklearn.naive_bayes import ComplementNB, MultinomialNB
        return (ComplementNB if name == "cnb" else MultinomialNB)(**{**dict(alpha=.3), **kw})
    if name in ("lgbm", "lgbm_reg", "lgbm_l1"):
        import lightgbm as lgb
        p = {**dict(n_estimators=500, learning_rate=.05, num_leaves=63, subsample=.8, subsample_freq=1, colsample_bytree=.5, random_state=seed, n_jobs=n_jobs, verbose=-1), **kw}
        return lgb.LGBMClassifier(**p) if name == "lgbm" else lgb.LGBMRegressor(objective="l1" if name == "lgbm_l1" else "l2", **p)
    if name == "xgb":
        from xgboost import XGBClassifier
        return XGBClassifier(**{**dict(n_estimators=500, learning_rate=.05, max_depth=6, tree_method="hist", subsample=.8, colsample_bytree=.5, random_state=seed), **kw})
    if name in ("rf", "rf_reg"):
        from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
        return (RandomForestClassifier if name == "rf" else RandomForestRegressor)(**{**dict(n_estimators=300, min_samples_leaf=2, n_jobs=n_jobs, random_state=seed), **kw})
    if name == "mlp":
        from sklearn.neural_network import MLPClassifier
        return MLPClassifier(**{**dict(hidden_layer_sizes=(256,), early_stopping=True, max_iter=100, random_state=seed), **kw})
    if name == "smote_lr":
        from imblearn.over_sampling import SMOTE
        from imblearn.pipeline import make_pipeline as imb
        return imb(SMOTE(k_neighbors=kw.get("k", 5), random_state=seed), LogisticRegression(max_iter=2000))
    if name in ("ovr", "chain"):
        base = LogisticRegression(**{**dict(C=4.0, max_iter=2000), **kw})
        if name == "ovr":
            from sklearn.multiclass import OneVsRestClassifier
            return OneVsRestClassifier(base, n_jobs=4)
        from sklearn.multioutput import ClassifierChain
        return ClassifierChain(base, order="random", random_state=seed)
    if name in ("ridge", "ridge_log"):
        m = Ridge(**{**dict(alpha=1.0, solver="sag", random_state=seed), **kw})
        if name == "ridge":
            return m
        from sklearn.compose import TransformedTargetRegressor
        return TransformedTargetRegressor(m, func=np.log1p, inverse_func=np.expm1)
    if name in ("dummy", "dummy_reg"):
        from sklearn.dummy import DummyClassifier, DummyRegressor
        return DummyClassifier(strategy="prior") if name == "dummy" else DummyRegressor(strategy=kw.get("strategy", "mean"))
    raise ValueError(f"model tidak dikenal: {name}")

# ===== jalankan =====
m_ = weak != -1
vec = make_vectorizer(VEC)
clf = make_model(MODEL).fit(vec.fit_transform([t for t, k in zip(texts, m_) if k]), weak[m_])
pred = clf.predict(vec.transform(texts))


## Nilai pada gold

Gold tidak pernah dipakai menulis aturan.

In [ ]:
import numpy as np
from sklearn.metrics import accuracy_score, average_precision_score, f1_score, hamming_loss, log_loss, precision_score, recall_score, roc_auc_score, top_k_accuracy_score

def cls_report(y, proba=None, pred=None):
    """proba: (n, k). pred otomatis argmax bila kosong."""
    y = np.asarray(y)
    pred = pred if pred is not None else np.asarray(proba).argmax(1)
    r = dict(acc=accuracy_score(y, pred), macro_f1=f1_score(y, pred, average="macro", zero_division=0), weighted_f1=f1_score(y, pred, average="weighted", zero_division=0),
             precision=precision_score(y, pred, average="macro", zero_division=0), recall=recall_score(y, pred, average="macro", zero_division=0))
    if proba is not None:
        p = np.asarray(proba)
        r["log_loss"] = log_loss(y, np.clip(p, 1e-6, 1 - 1e-6), labels=np.arange(p.shape[1]))
        if p.shape[1] == 2:
            r.update(f1_pos=f1_score(y, pred, zero_division=0), roc_auc=roc_auc_score(y, p[:, 1]), pr_auc=average_precision_score(y, p[:, 1]))
        else:
            r["roc_auc_ovr"] = roc_auc_score(y, p, multi_class="ovr", labels=np.arange(p.shape[1]))
            r["top2"] = top_k_accuracy_score(y, p, k=2, labels=np.arange(p.shape[1]))
    return {k: round(float(v), 4) for k, v in r.items()}

def multilabel_report(Y, pred):
    Y, pred = np.asarray(Y), np.asarray(pred)
    return {k: round(float(v), 4) for k, v in dict(micro_f1=f1_score(Y, pred, average="micro", zero_division=0), macro_f1=f1_score(Y, pred, average="macro", zero_division=0),
            samples_f1=f1_score(Y, pred, average="samples", zero_division=0), hamming=hamming_loss(Y, pred), subset_acc=accuracy_score(Y, pred)).items()}

# ===== jalankan =====
if GOLD in df:
    g = df[GOLD].to_numpy()
    ok = g != -1
    print("aturan", lf_accuracy(weak[ok], g[ok]), "\nclassifier", cls_report(g[ok], pred=pred[ok]))


## Output

Label per baris.

In [ ]:
df["pred"] = np.array(list(LEXICON))[pred]
df.to_csv("submission.csv", index=False)
